## Exercice - héritage de classe et IA "apprenante"


- En réutilisant le code proposer :
- Créer une classe mère ``Ia`` et deux classes filles ``IaSmart`` et ``IaDumb`` afin de mutualiser le code commun qu'ont ces IA (make_first_choice est la même pour les deux)





- Implémenter un jeu Pierre Feuille Ciseau sur modèle similaire:
    - Implémenter une classe Game
    - Implémenter une classe RandomPlayer (joueur qui joue au hasard)
    - La classe Game prend en argument deux instances de RandomPlayer à l'initialisation, qui joueront l'un contre l'autre
    - Implémenter une classe CissorPlayer qui ne joue que Ciseau
    - Implémente une classe LearningPlayer qu'on fera jouer contre une instance de CissorPlayer et qui "apprendra" au fur et à mesure des parties

In [ ]:
import random


# --- Monty Hall : hierarchie d'IA ---
class Ia:
    """Classe mere : le premier choix est commun a toutes les IA."""

    def make_first_choice(self, doors):
        return random.choice(doors)

    def make_final_choice(self, first_choice, remaining_door):
        raise NotImplementedError


class IaDumb(Ia):
    """Ne change jamais de porte."""

    def make_final_choice(self, first_choice, remaining_door):
        return first_choice


class IaSmart(Ia):
    """Change toujours de porte."""

    def make_final_choice(self, first_choice, remaining_door):
        return remaining_door


def play_one_game(ia):
    doors = [0, 1, 2]
    winning = random.choice(doors)
    first = ia.make_first_choice(doors)
    opened = random.choice([d for d in doors if d != first and d != winning])
    remaining = [d for d in doors if d != first and d != opened][0]
    final = ia.make_final_choice(first, remaining)
    return int(final == winning)


def simulate(ia, n_games=10_000):
    return sum(play_one_game(ia) for _ in range(n_games)) / n_games


print("IaDumb  (ne change jamais) :", simulate(IaDumb()))
print("IaSmart (change toujours)  :", simulate(IaSmart()))


In [ ]:
# --- Pierre Feuille Ciseau : joueur apprenant ---
COUPS = ["pierre", "feuille", "ciseau"]
BAT = {"pierre": "ciseau", "feuille": "pierre", "ciseau": "feuille"}


class RandomPlayer:
    def play(self):
        return random.choice(COUPS)

    def observe(self, coup_adverse):
        pass


class CissorPlayer(RandomPlayer):
    def play(self):
        return "ciseau"


class LearningPlayer(RandomPlayer):
    """Compte les coups adverses et joue ce qui bat le coup le plus frequent."""

    def __init__(self):
        self.compte = {c: 0 for c in COUPS}

    def play(self):
        coup_frequent = max(self.compte, key=self.compte.get)
        for coup, perd_contre in BAT.items():
            if perd_contre == coup_frequent:
                return coup
        return random.choice(COUPS)

    def observe(self, coup_adverse):
        self.compte[coup_adverse] += 1


class Game:
    def __init__(self, player_1, player_2):
        self.player_1 = player_1
        self.player_2 = player_2

    def play_round(self):
        c1, c2 = self.player_1.play(), self.player_2.play()
        self.player_1.observe(c2)
        self.player_2.observe(c1)
        if c1 == c2:
            return 0
        return 1 if BAT[c1] == c2 else 2

    def play_many(self, n_rounds=100):
        scores = {0: 0, 1: 0, 2: 0}
        for _ in range(n_rounds):
            scores[self.play_round()] += 1
        return scores


game = Game(LearningPlayer(), CissorPlayer())
print("Nuls / J1 (apprenant) / J2 (ciseau) :", game.play_many(100))
